In [7]:
# Szabo: Anakinra
import pandas as pd
from pathlib import Path

# ===== File paths =====
data_week0_file = "raw_data/Szabo/anakinra_transfer_iu_with_barcode.csv"
aliquot_file = "aliquot_DCC_data_release_v2-0-10_shipped_mapped.tsv"
output_file = "molecular-test_szabo_DCC_data_release_v2-0-12.tsv"
qc_file = "QC_molecular-test_szabo_DCC_data_release_v2-0-12.tsv"

# ===== Read input data =====
df_data = pd.read_csv(data_week0_file, dtype=str)  # Source CSV
df_aliquot = pd.read_csv(aliquot_file, sep="\t", dtype=str)  # Aliquot mapping

# ===== Clean column names =====
df_data.columns = df_data.columns.str.strip()
df_aliquot.columns = df_aliquot.columns.str.strip()

# ===== Check required columns =====
required_data_cols = {"Barcode", "Specimen_Type", "Serum IL-1RA level (pg/mL)"}
missing_data = required_data_cols - set(df_data.columns)
if missing_data:
    raise ValueError(f"Missing columns in source data: {missing_data}")

required_ali_cols = {"*submitter_id", "*follow_ups.submitter_id"}
missing_ali = required_ali_cols - set(df_aliquot.columns)
if missing_ali:
    raise ValueError(f"Missing columns in aliquot file: {missing_ali}")

# ===== Remove empty barcodes and trim =====
df_data = df_data[df_data["Barcode"].notna()].copy()
df_data["Barcode"] = df_data["Barcode"].astype(str).str.strip()

df_aliquot = df_aliquot[df_aliquot["*submitter_id"].notna()].copy()
df_aliquot["*submitter_id"] = df_aliquot["*submitter_id"].astype(str).str.strip()
df_aliquot["*follow_ups.submitter_id"] = df_aliquot["*follow_ups.submitter_id"].astype(str).str.strip()

# ===== Remove duplicate aliquot IDs =====
df_aliquot = df_aliquot.drop_duplicates(subset=["*submitter_id"], keep="first")

# ===== Create lookup =====
aliquot_lookup = df_aliquot.set_index("*submitter_id")

# ===== Define columns for mapping =====
value_col = "Serum IL-1RA level (pg/mL)"
lab_test_col = "Specimen_Type"

# ===== Output lists =====
output_rows = []
qc_rows = []
source_name = Path(data_week0_file).name

# ===== Process rows =====
for _, row in df_data.iterrows():
    barcode = (row.get("Barcode") or "").strip()
    if not barcode:
        qc_rows.append({"source_file": source_name, "Barcode": "", "reason": "Empty barcode"})
        continue

    if barcode in aliquot_lookup.index:
        ali_row = aliquot_lookup.loc[barcode]
        followup_id = ali_row["*follow_ups.submitter_id"]

        new_submitter_id = f"{followup_id}_szabo"

        output_rows.append({
            "*type": "molecular_test",
            "project_id": "ARDaC-AlcHepNet",
            "*submitter_id": new_submitter_id,
            "aliquots.submitter_id": barcode,
            "*follow_ups.submitter_id": followup_id,
            "blood_test_normal_range_lower": "",
            "blood_test_normal_range_upper": "",
            "days_to_test": "",
            "gene_symbol": "",
            "laboratory_test": row.get(lab_test_col, ""),
            "molecular_analysis_method": "",
            "test_result": "",
            "test_unit": "pg/mL",
            "test_value": row.get(value_col, "")
        })
    else:
        qc_rows.append({"source_file": source_name, "Barcode": barcode, "reason": "Barcode not found in aliquot *submitter_id"})

# ===== Save output =====
output_df = pd.DataFrame(output_rows)
cols_order = [
    "*type", "project_id", "*submitter_id", "aliquots.submitter_id", "*follow_ups.submitter_id",
    "blood_test_normal_range_lower", "blood_test_normal_range_upper", "days_to_test", "gene_symbol",
    "laboratory_test", "molecular_analysis_method", "test_result", "test_unit", "test_value"
]
for c in cols_order:
    if c not in output_df.columns:
        output_df[c] = ""
output_df = output_df[cols_order]
output_df.to_csv(output_file, sep="\t", index=False)

# ===== Save QC =====
qc_df = pd.DataFrame(qc_rows, columns=["source_file", "Barcode", "reason"])
qc_df.to_csv(qc_file, sep="\t", index=False)

print(f"Records generated: {len(output_df)}; QC records: {len(qc_df)}")

Records generated: 993; QC records: 0
